In [1]:
import os
import shutil
import random
from pathlib import Path

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
DATASET_ROOT = r"D:\Mold Studies Binary\Mould Binary Classification"

TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15

RANDOM_SEED  = 27
IMG_EXTS     = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

PROTECTED    = {"train", "val", "test"}

# ─────────────────────────────────────────────
# STEP 1 — Discover classes
# ─────────────────────────────────────────────
dataset_path = Path(DATASET_ROOT)

classes = sorted([
    d.name for d in dataset_path.iterdir()
    if d.is_dir() and d.name not in PROTECTED
])

print(f"✅ Found {len(classes)} classes:\n")
for i, cls in enumerate(classes, 1):
    n_images = sum(
        1 for f in (dataset_path / cls).iterdir()
        if f.is_file() and f.suffix.lower() in IMG_EXTS
    )
    print(f"  {i:>2}. {cls:<15} {n_images:>6} images")

# ─────────────────────────────────────────────
# STEP 2 — Validate ratios
# ─────────────────────────────────────────────
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6, \
    "Train + Val + Test ratios must sum to 1.0"

# ─────────────────────────────────────────────
# STEP 3 — Split and copy files
# ─────────────────────────────────────────────
random.seed(RANDOM_SEED)

split_summary = []

for cls in classes:
    src_folder = dataset_path / cls

    images = [
        f for f in src_folder.iterdir()
        if f.is_file() and f.suffix.lower() in IMG_EXTS
    ]

    if not images:
        print(f"⚠️  No images found in class: {cls}")
        continue

    random.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * TRAIN_RATIO)
    n_val   = int(n_total * VAL_RATIO)
    n_test  = n_total - n_train - n_val

    splits = {
        "train": images[:n_train],
        "val"  : images[n_train : n_train + n_val],
        "test" : images[n_train + n_val:]
    }

    for split_name, split_files in splits.items():
        dest_folder = dataset_path / split_name / cls
        dest_folder.mkdir(parents=True, exist_ok=True)

        for img_path in split_files:
            shutil.copy2(img_path, dest_folder / img_path.name)

    split_summary.append({
        "class" : cls,
        "total" : n_total,
        "train" : n_train,
        "val"   : n_val,
        "test"  : n_test,
    })

# ─────────────────────────────────────────────
# STEP 4 — Summary table
# ─────────────────────────────────────────────
print(f"\n{'─'*55}")
print(f"{'Class':<15} {'Total':>7} {'Train':>7} {'Val':>6} {'Test':>6}")
print(f"{'─'*55}")

for row in split_summary:
    print(f"{row['class']:<15} {row['total']:>7} "
          f"{row['train']:>7} {row['val']:>6} {row['test']:>6}")

totals = {k: sum(r[k] for r in split_summary)
          for k in ["total", "train", "val", "test"]}
print(f"{'─'*55}")
print(f"{'TOTAL':<15} {totals['total']:>7} "
      f"{totals['train']:>7} {totals['val']:>6} {totals['test']:>6}")

# ─────────────────────────────────────────────
# STEP 5 — Delete original class folders
# ─────────────────────────────────────────────
print("\n🗑️  Removing original class folders...")

for cls in classes:
    if cls.lower() in PROTECTED:
        print(f"   Skipped (protected): {cls}")
        continue
    original_folder = dataset_path / cls
    if original_folder.exists():
        shutil.rmtree(original_folder)
        print(f"   Deleted: {original_folder}")

print(f"\n✅ Split complete. Output saved to:")
print(f"   {DATASET_ROOT}")
print(f"\n   Structure:")
print(f"   {DATASET_ROOT}/")
print(f"   ├── train/")
print(f"   │   ├── mold/")
print(f"   │   └── no mold/")
print(f"   ├── val/")
print(f"   │   ├── mold/")
print(f"   │   └── no mold/")
print(f"   └── test/")
print(f"       ├── mold/")
print(f"       └── no mold/")

✅ Found 2 classes:

   1. Mold              1841 images
   2. No mold           2480 images

───────────────────────────────────────────────────────
Class             Total   Train    Val   Test
───────────────────────────────────────────────────────
Mold               1841    1288    276    277
No mold            2480    1736    372    372
───────────────────────────────────────────────────────
TOTAL              4321    3024    648    649

🗑️  Removing original class folders...
   Deleted: D:\Mold Studies Binary\Mould Binary Classification\Mold
   Deleted: D:\Mold Studies Binary\Mould Binary Classification\No mold

✅ Split complete. Output saved to:
   D:\Mold Studies Binary\Mould Binary Classification

   Structure:
   D:\Mold Studies Binary\Mould Binary Classification/
   ├── train/
   │   ├── mold/
   │   └── no mold/
   ├── val/
   │   ├── mold/
   │   └── no mold/
   └── test/
       ├── mold/
       └── no mold/
